# Testing, debugging and logging
* so far we always tested our code manually, by running it and looking if the output "looks right" - this doesnt scale for bigger projects
* automated tests catch bugs BEFORE they reach real usage (or worse, production), give us confidence when refactoring later, and act as living documentation of how our code is supposed to behave
* debugging tools help us figure out WHY something goes wrong, instead of just guessing and adding random print() statements everywhere
* logging is the "grown up" version of print() - it gives us severity levels, timestamps, and the ability to turn parts of it on/off without touching the code
* in this lesson we cover the builtin `unittest` module, the builtin `logging` module, and a short overview of debugging tools (`pdb`/`breakpoint()` and `traceback`)

## why automated tests matter
* they catch bugs before they reach real usage - a test fails on YOUR machine, not in front of a customer
* they give you confidence when refactoring - if you change how a function works internally, the tests tell you immediately if you broke something
* they serve as living documentation - a test literally shows an example of "given this input, we expect this output", often clearer than a comment
* writing tests feels like extra work at first, but it pays off fast once a project grows beyond a "throwaway script" size

## unittest - pythons builtin testing module
* `unittest` is part of the standard library, no pip install needed
* it uses a class-based style (often called "xUnit style", we will see why in the comparison section later)
* the basic recipe: write a `class SomethingTest(unittest.TestCase):`, and inside it a bunch of `def test_...(self):` methods
* each test method uses assert-methods like `self.assertEqual(a, b)`, `self.assertTrue(x)`, `self.assertRaises(SomeError)` instead of the plain `assert` keyword we saw in lesson 06 - this way unittest can collect ALL failures nicely instead of stopping at the first one
* lets first define a small function we want to test - similar to the `is_prime` function from lesson 03

In [1]:
# a small function to test - returns True if the number is prime, raises an error for invalid input
def is_prime(number):
    # we reuse the "raise our own exception for bad input" pattern from lesson 06
    if not isinstance(number, int):
        raise TypeError(f"is_prime needs an int, got {type(number).__name__}")  # bad input --> fail loudly instead of silently wrong result
    if number < 2:
        return False  # 0, 1 and negative numbers are not prime by definition
    for divisor in range(2, number):
        if number % divisor == 0:
            return False  # found a divisor --> not prime
    else:
        return True  # for...else - loop finished completely without finding a divisor --> prime

# quick manual sanity check before we even write real tests
print(is_prime(7))
print(is_prime(8))

True
False


## writing our first TestCase
* every test class inherits from `unittest.TestCase` - this is what gives us access to all the `self.assertX` methods
* every test METHOD name must start with `test_` - thats how unittest discovers which methods to actually run as tests (a helper method not starting with `test_` gets ignored)
* `self.assertEqual(a, b)` checks a == b, `self.assertTrue(x)`/`self.assertFalse(x)` checks truthiness, `self.assertRaises(SomeError)` (used as a context manager with `with`) checks that the code block inside raises exactly that exception type - this ties directly back to the exception handling we learned in lesson 06

In [2]:
import unittest  # standard library, no install needed

class IsPrimeTest(unittest.TestCase):
    # each test method gets a fresh instance of the class, but they all share the class name as grouping

    def test_known_primes(self):
        # assertTrue checks that the given expression evaluates to True
        self.assertTrue(is_prime(2))
        self.assertTrue(is_prime(13))

    def test_known_non_primes(self):
        # assertFalse is the opposite of assertTrue
        self.assertFalse(is_prime(1))
        self.assertFalse(is_prime(4))

    def test_exact_value(self):
        # assertEqual is the most used assert - it compares two values for equality and gives a nice diff on failure
        self.assertEqual(is_prime(9), False)
        self.assertEqual(is_prime(11), True)

    def test_bad_input_raises_typeerror(self):
        # assertRaises used as a "with" context manager - the code inside the block MUST raise this exact exception
        # if it doesnt raise (or raises something else), the test fails - this is how we test our own raise-statements from lesson 06
        with self.assertRaises(TypeError):
            is_prime("not a number")

# IMPORTANT: unittest.main() normally calls sys.exit() when it finishes, which would kill our whole notebook kernel!
# inside a notebook (or any interactive session) we always pass argv=["ignored"] and exit=False to avoid that
unittest.main(argv=["ignored"], exit=False, verbosity=2)

test_bad_input_raises_typeerror (__main__.IsPrimeTest.test_bad_input_raises_typeerror) ... 

ok


test_exact_value (__main__.IsPrimeTest.test_exact_value) ... 

ok


test_known_non_primes (__main__.IsPrimeTest.test_known_non_primes) ... 

ok


test_known_primes (__main__.IsPrimeTest.test_known_primes) ... 

ok


----------------------------------------------------------------------
Ran 4 tests in 0.004s

OK


## setUp and tearDown - shared fixture code
* `setUp(self)` runs automatically BEFORE every single test method in the class
* `tearDown(self)` runs automatically AFTER every single test method in the class (even if the test failed!)
* this is super useful for shared setup work, like resetting a small in-memory object to a known starting state before each test, so tests dont accidentally influence each other

In [3]:
class Counter:
    # a tiny stateful class we want to test - counting up and resetting
    def __init__(self):
        self.value = 0

    def increment(self):
        self.value += 1

class CounterTest(unittest.TestCase):

    def setUp(self):
        # this runs before EVERY test method below - guarantees each test starts with a fresh, known counter
        print("setUp: creating a fresh Counter")
        self.counter = Counter()  # self.counter is now available in every test method

    def tearDown(self):
        # this runs after EVERY test method, useful for cleanup (closing files, connections, resetting global state, ...)
        print("tearDown: test finished")

    def test_starts_at_zero(self):
        self.assertEqual(self.counter.value, 0)

    def test_increment_once(self):
        self.counter.increment()
        self.assertEqual(self.counter.value, 1)

    def test_increment_three_times(self):
        # if setUp did NOT run fresh for every test, this test could accidentally see leftover state from another test
        self.counter.increment()
        self.counter.increment()
        self.counter.increment()
        self.assertEqual(self.counter.value, 3)

unittest.main(argv=["ignored"], exit=False, verbosity=2)

test_increment_once (__main__.CounterTest.test_increment_once) ... 

ok


test_increment_three_times (__main__.CounterTest.test_increment_three_times) ... 

ok


test_starts_at_zero (__main__.CounterTest.test_starts_at_zero) ... 

...

ok


test_known_primes (__main__.IsPrimeTest.test_known_primes) ... 

ok


----------------------------------------------------------------------
Ran 7 tests in 0.007s

OK


setUp: creating a fresh Counter
tearDown: test finished
setUp: creating a fresh Counter
tearDown: test finished
setUp: creating a fresh Counter
tearDown: test finished


## test organization in real projects
* in a real project you dont just dump test code into the same notebook/file as the actual code - common conventions are:
* a separate top-level `tests/` folder, next to (not inside) your actual package/module code
* one test file per module being tested, following the naming convention `test_<module_name>.py` (e.g. `math_utils.py` gets tested by `tests/test_math_utils.py`)
* test runners (both `unittest` and `pytest`) automatically discover any file matching `test_*.py` and any class/function starting with `test`/`Test`, so sticking to the naming convention means you rarely have to configure anything manually
* we wont actually create this folder structure here, but keep it in mind for your own projects

## pytest - the most popular third party alternative
* `pytest` is NOT part of the standard library (needs `pip install pytest`) but is by far the most widely used testing tool in the real python world
* instead of `self.assertEqual(a, b)` you just write a plain `assert a == b` - pytest rewrites assert statements internally to still give you nice failure messages
* no `TestCase` class or `self` needed - a pytest test is often just a plain function starting with `test_`
* "fixtures" (pytest's version of setUp/tearDown) work via dependency injection - you declare a fixture as a function parameter and pytest automatically provides it
* huge plugin ecosystem (coverage reports, parallel test running, mocking helpers, and much more)
* we only describe it here conceptually - we are not installing/running it in this lesson, `unittest` is all we need since it ships with python itself

```python
# what an equivalent pytest test would roughly look like (not executed here, just for comparison):
def test_known_primes():
    assert is_prime(2)
    assert is_prime(13)

def test_bad_input_raises_typeerror():
    import pytest
    with pytest.raises(TypeError):
        is_prime("not a number")
```

## why print() is not a great tool for real applications
* `print()` always shows up, there is no concept of severity - you cant say "only show me the important stuff"
* no easy way to turn it off or redirect it somewhere else (a file, a monitoring system, ...) without editing every single print statement
* no timestamps by default - hard to tell WHEN something happened, especially in a long-running program
* the `logging` module (also standard library!) solves all of this properly

## logging basics
* `logging.basicConfig(level=...)` configures the root logging setup once, usually at the start of a program
* there are 5 standard severity levels, from least to most severe: `DEBUG` (detailed diagnostic info), `INFO` (general confirmation things are working), `WARNING` (something unexpected but not breaking), `ERROR` (something failed), `CRITICAL` (very serious, program might not be able to continue)
* the configured level is a FILTER - only messages at that level or MORE severe actually get shown, everything less severe is silently dropped
* `logger = logging.getLogger(__name__)` is the standard pattern - it creates/gets a logger named after the current module, which helps a lot when you have logs coming from many different files

In [4]:
import logging

# configure the root logger once - level=INFO means: show INFO and everything more severe (WARNING, ERROR, CRITICAL), but hide DEBUG
# force=True is needed here because notebooks sometimes already configure logging behind the scenes, this makes sure our config actually applies
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s", force=True)

# __name__ is the module name - in a notebook this is usually "__main__", in a real project it would be e.g. "myproject.utils"
logger = logging.getLogger(__name__)

# lets log a message at every single level and see which ones actually make it through
logger.debug("this is a DEBUG message - very detailed, for developers only")      # filtered out, level is INFO
logger.info("this is an INFO message - normal confirmation that things work")     # shown
logger.warning("this is a WARNING message - something looks off but we continue") # shown
logger.error("this is an ERROR message - something actually failed")             # shown
logger.critical("this is a CRITICAL message - very serious problem")             # shown

# notice DEBUG did not print at all - because we configured level=INFO, anything less severe than INFO gets silently dropped
# this means we can leave logger.debug(...) calls scattered all over our code for future debugging, without spamming normal users

2026-09-26 10:52:44,567 INFO __main__: this is an INFO message - normal confirmation that things work


2026-09-26 10:52:44,568 WARNING __main__: this is a WARNING message - something looks off but we continue


2026-09-26 10:52:44,568 ERROR __main__: this is an ERROR message - something actually failed


2026-09-26 10:52:44,569 CRITICAL __main__: this is a CRITICAL message - very serious problem


## lazy formatting in log calls
* you CAN write an f-string directly inside a log call, like `logger.info(f"value is {value}")` - but this is not ideal
* the preferred way is the lazy `%s` style: `logger.info("value is %s", value)`
* WHY: with an f-string, python has to build the formatted string FIRST, before even calling `logger.info(...)` - even if that log message ends up being filtered out and never shown!
* with the lazy `%s` style, the string is only actually formatted internally if the log level is enabled - if the message would be filtered out anyway, python skips the formatting work entirely
* for a single log call this makes basically no difference, but in a hot loop that calls `logger.debug(...)` thousands of times, this saves real performance when DEBUG logging is turned off

In [5]:
# lets demonstrate both styles side by side (both print the same text here, since INFO is enabled)
value = 42

# f-string style - value gets formatted into the string immediately, no matter what
logger.info(f"(f-string style) value is {value}")

# lazy %s style - logging only formats "value is %s" % value internally IF the INFO level is actually enabled
logger.info("(lazy %%s style) value is %s", value)

# now lets bump the level up to WARNING and try a debug call in both styles - neither should print, but the lazy one is cheaper to skip
logging.getLogger(__name__).setLevel(logging.WARNING)
logger.debug(f"(f-string style, expensive) computed value is {value ** 2}")  # value ** 2 gets computed even though this line prints nothing
logger.debug("(lazy style, cheap) computed value is %s", value ** 2)         # value ** 2 still gets computed in THIS demo since we pass it as an argument, but the string formatting itself is skipped
print("both debug calls above produced no output, because we raised the level to WARNING")

# reset back to INFO for the rest of the lesson
logger.setLevel(logging.INFO)

2026-09-26 10:52:44,576 INFO __main__: (f-string style) value is 42


2026-09-26 10:52:44,577 INFO __main__: (lazy %s style) value is 42


both debug calls above produced no output, because we raised the level to WARNING


## debugging tools overview
* sometimes reading the code and adding print()/logging isnt enough, and we need to pause execution and poke around live
* python 3.7+ has a builtin `breakpoint()` function - calling it drops you into `pdb` (the Python DeBugger), an interactive command line debugger, right at that exact line
* IMPORTANT: we will NOT actually call `breakpoint()` in this notebook - doing so in a non-interactive/headless environment would just hang forever waiting for input that never comes! we only show it commented-out below
* once inside pdb, a few commands you'll use constantly:
    * `n` (next) - run the current line, step to the next one (does not step INTO function calls)
    * `s` (step) - step INTO a function call, for when you want to dig deeper
    * `c` (continue) - resume normal execution until the next breakpoint (or the end)
    * `p variable` (print) - print the current value of a variable
    * `l` (list) - show the source code around the current line
    * `q` (quit) - abort the debugging session entirely

In [6]:
def calculate_total(prices):
    total = 0
    for price in prices:
        # breakpoint()  # uncomment this line locally to pause execution here and inspect variables with pdb
        total += price
    return total

print(calculate_total([10, 20, 30]))
# if you uncomment the breakpoint() above and run this LOCALLY in a real terminal/IDE (not here),
# execution pauses right there and drops you into an interactive (Pdb) prompt where you can type n, s, c, p total, l, q, ...

60


## the traceback module
* we already know `try`/`except` from lesson 06 to catch and handle exceptions gracefully
* sometimes we want to still SEE the full traceback (the detailed "where exactly did this go wrong" info) even though we ARE handling the exception and not letting the program crash
* `traceback.print_exc()`, called inside an `except` block, prints the full traceback of the currently-handled exception without re-raising it - great for logging unexpected errors while still recovering gracefully

In [7]:
import traceback

def risky_calculation(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        print("--- something went wrong, here is the full traceback for debugging purposes ---")
        traceback.print_exc()  # prints the whole traceback (file, line, call stack) to stderr, without stopping the program
        print("--- end of traceback, program continues normally ---")
        return None  # we still recover gracefully and return a safe fallback value

result = risky_calculation(10, 0)
print("result after the handled error:", result)
print("the program is still running just fine!")

--- something went wrong, here is the full traceback for debugging purposes ---
--- end of traceback, program continues normally ---
result after the handled error: None
the program is still running just fine!


Traceback (most recent call last):
  File "/tmp/ipykernel_7751/1367666330.py", line 5, in risky_calculation
    return a / b
           ~~^~~
ZeroDivisionError: division by zero


## quick comparison to other languages
* python's `unittest` (xUnit-style: a `TestCase` class, `self.assertX` methods) is structurally almost identical to Java's JUnit, C#'s NUnit/MSTest, and PHP's PHPUnit - all of these descend from the same original "xUnit" design pattern
* `pytest`'s plain-`assert`-based style is closer in spirit to JavaScript's Jest/Mocha (`expect(x).toBe(y)` vs pytest's `assert x == y`) - both are far less boilerplate-heavy than classic xUnit-style testing
* python's `logging` module design (loggers, handlers, levels, formatters) is very similar to Java's `java.util.logging`/Log4j/SLF4J and to Node.js's popular `winston`/`pino` libraries - hierarchical named loggers with configurable levels and handlers is a shared pattern across basically every ecosystem
* python's `pdb`/`breakpoint()` corresponds to JavaScript's `debugger` statement + browser devtools, or an IDE's builtin debugger/breakpoints in Java/C#/C++ - on the C/C++ command line the equivalent tool is `gdb`

## Exercises
1. write a function `divide(a, b)` and a `unittest.TestCase` with tests for a normal division, and for a `ZeroDivisionError` using `assertRaises`
2. add `setUp`/`tearDown` methods to a small `TestCase` that tracks a shared counter, printing a message whenever each one runs
3. configure logging at `INFO` level and log one message at each of the 5 standard levels, observe which ones actually get printed
4. write a function that catches an exception internally, logs it with `logging.error`, and still returns a safe default value instead of crashing

In [8]:
# TODO: solve exercise 1 here (divide + TestCase with assertRaises)


# TODO: solve exercise 2 here (setUp/tearDown tracking a shared counter)


# TODO: solve exercise 3 here (logging at INFO level, one message per level)


# TODO: solve exercise 4 here (catch exception, logging.error, return safe default)

### sample solutions
* try the exercises above yourself first, only look here if you are stuck or want to compare your approach

In [9]:
# sample solution 1 - divide + TestCase
def divide(a, b):
    return a / b

class DivideTest(unittest.TestCase):
    def test_normal_division(self):
        self.assertEqual(divide(10, 2), 5)

    def test_division_by_zero_raises(self):
        with self.assertRaises(ZeroDivisionError):
            divide(10, 0)

unittest.main(argv=["ignored"], exit=False, verbosity=2)

# sample solution 2 - setUp/tearDown tracking a shared counter
class ExerciseCounterTest(unittest.TestCase):
    def setUp(self):
        print("setUp running - resetting counter to 0")
        self.counter = 0

    def tearDown(self):
        print("tearDown running - counter ended at", self.counter)

    def test_increment(self):
        self.counter += 1
        self.assertEqual(self.counter, 1)

    def test_increment_twice(self):
        self.counter += 1
        self.counter += 1
        self.assertEqual(self.counter, 2)

unittest.main(argv=["ignored"], exit=False, verbosity=2)

# sample solution 3 - logging at INFO level, one message per level
logging.basicConfig(level=logging.INFO, force=True)
exercise_logger = logging.getLogger("exercise3")
exercise_logger.setLevel(logging.INFO)
exercise_logger.debug("debug message - should NOT print, below INFO level")
exercise_logger.info("info message - should print")
exercise_logger.warning("warning message - should print")
exercise_logger.error("error message - should print")
exercise_logger.critical("critical message - should print")

# sample solution 4 - catch exception, log it, return safe default
def safe_average(numbers):
    error_logger = logging.getLogger("exercise4")
    try:
        return sum(numbers) / len(numbers)
    except ZeroDivisionError:
        error_logger.error("cant calculate average of an empty list, returning 0 instead")
        return 0  # safe fallback value instead of crashing

print(safe_average([2, 4, 6]))
print(safe_average([]))

test_increment_once (__main__.CounterTest.test_increment_once) ... 

ok


test_increment_three_times (__main__.CounterTest.test_increment_three_times) ... 

ok


test_starts_at_zero (__main__.CounterTest.test_starts_at_zero) ... 

...

INFO:exercise3:info message - should print


ERROR:exercise3:error message - should print


CRITICAL:exercise3:critical message - should print


ERROR:exercise4:cant calculate average of an empty list, returning 0 instead


setUp: creating a fresh Counter
tearDown: test finished
setUp: creating a fresh Counter
tearDown: test finished
setUp: creating a fresh Counter
tearDown: test finished
setUp: creating a fresh Counter
tearDown: test finished
setUp: creating a fresh Counter
tearDown: test finished
setUp: creating a fresh Counter
tearDown: test finished
setUp running - resetting counter to 0
tearDown running - counter ended at 1
setUp running - resetting counter to 0
tearDown running - counter ended at 2
4.0
0


* congratulation you finished this lesson

## what we learned
* why automated tests matter: catching bugs early, enabling confident refactoring, and acting as living documentation
* writing `unittest.TestCase` classes with `test_` methods and `self.assertEqual`/`assertTrue`/`assertFalse`/`assertRaises`
* running unittest safely inside a notebook with `unittest.main(argv=["ignored"], exit=False, verbosity=2)` to avoid killing the kernel
* `setUp`/`tearDown` for shared fixture code that runs before/after every test method
* common real-project test organization: a `tests/` folder, `test_*.py` naming convention
* `pytest` as the most popular third-party alternative - plain `assert`, fixtures via dependency injection, big plugin ecosystem
* why `logging` beats `print()` for real applications, `basicConfig`, the 5 standard severity levels, and the `logging.getLogger(__name__)` pattern
* lazy `%s` formatting in log calls vs. f-strings, and why it saves performance when a log level is filtered out
* debugging tools: `breakpoint()`/`pdb` and its core commands (`n`, `s`, `c`, `p`, `l`, `q`), and `traceback.print_exc()` for inspecting a handled exception without crashing